# Module 05 — Neural Network Foundations
## 05-10: Complete MLP Pipeline

**Objective:** Build a production-quality MLP training pipeline end-to-end —
data ingestion, architecture search, full training with regularisation and LR
scheduling, comprehensive evaluation, and ONNX export.

**Prerequisites:** 05-02 (MLP architecture); 05-05 (Regularisation — Dropout,
BatchNorm, early stopping); 05-08 (Weight Initialisation — He/Kaiming);
05-09 (Optimisers — AdamW, LR schedules).


## Part 0 — Setup & Prerequisites

This capstone notebook applies every tool from Module 05 in a single coherent
pipeline.  Nothing new is derived here — instead, we make *engineering decisions*
based on the empirical evidence gathered in the earlier notebooks:

| Decision | Evidence from |
|---|---|
| He normal init | 05-08 (best for ReLU at all depths) |
| AdamW optimiser | 05-09 (best convergence + weight control) |
| Dropout regularisation | 05-05 (reduces overfitting) |
| ReLU activation | 05-03 (beats tanh at depth for classification) |
| Cross-entropy loss | 05-04 (correct loss for multi-class) |

> **Concept ownership:** ONNX export and inference profiling are introduced here.
> Learning-rate scheduling (beyond cosine annealing) is covered in **Module 16**.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import warnings
import pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections.abc import Callable

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score,
)

import random
import math
print(f"Python:    {sys.version.split()[0]}")
print(f"PyTorch:   {torch.__version__}")
print(f"NumPy:     {np.__version__}")
print(f"sklearn:   ", end="")
import sklearn; print(sklearn.__version__)
if torch.cuda.is_available():
    print(f"CUDA:      {torch.version.cuda}")
    print(f"GPU:       {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility & Device ──────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# FashionMNIST
FASHION_MEAN    = (0.2860,)
FASHION_STD     = (0.3530,)
NUM_CLASSES     = 10
INPUT_DIM       = 784       # 28 × 28
FASHION_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

# Architecture search (fast pass on subset)
SEARCH_EPOCHS  = 3
SEARCH_TRAIN_N = 10_000    # samples used per search trial

# Full training
BATCH_SIZE    = 256
NUM_EPOCHS    = 25
LEARNING_RATE = 3e-4       # AdamW default for Transformers / MLPs
WEIGHT_DECAY  = 1e-4
DROPOUT_RATE  = 0.3
PATIENCE      = 5          # early-stopping patience (epochs)
ETA_MIN       = 1e-6       # minimum LR for cosine scheduler

# ONNX export
ONNX_PATH     = "modules/module_05/fashionmnist_mlp.onnx"


### Data — FashionMNIST

We use FashionMNIST's **official train (60 000) / test (10 000)** splits and
take a **90 / 10** split of the training set for train/val.  Images are normalised
to zero mean and unit variance.


In [ ]:
# ── FashionMNIST pipeline ─────────────────────────────────────────────────────
transform_f = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(FASHION_MEAN, FASHION_STD),
])

full_train_f = torchvision.datasets.FashionMNIST(
    root="data/", train=True,  download=True, transform=transform_f,
)
test_set_f = torchvision.datasets.FashionMNIST(
    root="data/", train=False, download=True, transform=transform_f,
)

generator_f   = torch.Generator().manual_seed(SEED)
train_size_f  = int(0.9 * len(full_train_f))
val_size_f    = len(full_train_f) - train_size_f
train_set_f, val_set_f = torch.utils.data.random_split(
    full_train_f, [train_size_f, val_size_f], generator=generator_f,
)

train_loader = DataLoader(train_set_f, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set_f,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set_f,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_set_f):,}  Val: {len(val_set_f):,}  Test: {len(test_set_f):,}")

# ── EDA: sample images per class ──────────────────────────────────────────────
n_per_class = 5
fig_eda, axes_eda = plt.subplots(NUM_CLASSES, n_per_class,
                                  figsize=(n_per_class * 1.5, NUM_CLASSES * 1.5))
rng_eda    = np.random.default_rng(SEED)
seen_count = np.zeros(NUM_CLASSES, dtype=int)
needed     = np.full(NUM_CLASSES, n_per_class, dtype=int)

for img_e, label_e in full_train_f:
    cls_e = int(label_e)
    if seen_count[cls_e] < n_per_class:
        ax_e = axes_eda[cls_e, seen_count[cls_e]]
        ax_e.imshow(img_e.squeeze(), cmap="gray")
        ax_e.axis("off")
        if seen_count[cls_e] == 0:
            ax_e.set_ylabel(FASHION_CLASSES[cls_e], fontsize=7, rotation=0,
                            labelpad=55, va="center")
        seen_count[cls_e] += 1
    if np.all(seen_count >= n_per_class):
        break

plt.suptitle("FashionMNIST — 5 samples per class", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── EDA: class distribution bar chart ─────────────────────────────────────────
all_labels  = [int(label) for _, label in full_train_f]
class_counts = np.bincount(all_labels)
fig_cc, ax_cc = plt.subplots(figsize=(10, 3))
bar_cc = ax_cc.bar(range(NUM_CLASSES), class_counts, color=plt.cm.tab10(
    np.linspace(0, 1, NUM_CLASSES)), edgecolor="k", lw=0.6)
ax_cc.set_xticks(range(NUM_CLASSES))
ax_cc.set_xticklabels(FASHION_CLASSES, rotation=30, ha="right", fontsize=9)
ax_cc.set_ylabel("Count", fontsize=11)
ax_cc.set_title("Class Distribution — FashionMNIST (Training Set)",
                fontsize=11, fontweight="bold")
ax_cc.grid(axis="y", alpha=0.3)
for bar_c, cnt_c in zip(bar_cc, class_counts):
    ax_cc.text(bar_c.get_x() + bar_c.get_width() / 2.0,
               bar_c.get_height() + 50, str(cnt_c),
               ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()
print(f"Dataset is {'balanced' if class_counts.std() < 100 else 'imbalanced'} "
      f"(std={class_counts.std():.0f}, min={class_counts.min()}, max={class_counts.max()})")


---
## Part 1 — Architecture Search

Rather than guessing a fixed architecture, we run a **quick 3-epoch trial** for
each candidate on a 10 000-sample training subset.  This reveals which depth/width
gives the best validation accuracy before committing to a full 25-epoch run.

### Search Space

| Architecture | Hidden layers | Params (approx.) |
|---|---|---|
| Tiny | [128] | ~102 K |
| Small | [256, 128] | ~234 K |
| Medium | [512, 256] | ~533 K |
| Deep-3 | [256, 256, 256] | ~464 K |
| Deep-wide | [512, 256, 128] | ~533 K |
| XL | [1024, 512, 256] | ~2.1 M |

All candidates use **Dropout(0.3)** and **He initialisation** for consistency.


### Part 1 — Architecture & Regularisation Search

We build a configurable  class and sweep over hidden dimensions, depths, and regularisation options (Dropout, BatchNorm) to find the best architecture via a fast 3-epoch search on a subset of the data.

In [ ]:
# ── FlexMLP: configurable MLP with optional BatchNorm and Dropout ─────────────

class FlexMLP(nn.Module):
    '''Configurable MLP for FashionMNIST classification.

    Supports arbitrary depth/width, optional BatchNorm, Dropout, and
    multiple activation functions.

    Attributes:
        net: Sequential stack built from the given configuration.
    '''

    def __init__(
        self,
        layer_sizes:   list[int],
        dropout_rate:  float = 0.0,
        use_batchnorm: bool  = False,
        activation:    str   = "relu",
    ) -> None:
        '''Build FlexMLP.

        Args:
            layer_sizes:   [n_in, h1, h2, ..., n_out] layer widths.
            dropout_rate:  Dropout probability (0 = disabled).
            use_batchnorm: Insert BatchNorm1d after each hidden Linear.
            activation:    "relu", "tanh", or "gelu".
        '''
        super().__init__()
        act_map: dict[str, type] = {
            "relu": nn.ReLU,
            "tanh": nn.Tanh,
            "gelu": nn.GELU,
        }
        if activation not in act_map:
            raise ValueError(f"Unknown activation '{activation}'. "
                             f"Choose from {list(act_map)}.")
        act_fn = act_map[activation]

        layers: list[nn.Module] = []
        for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            layers.append(nn.Linear(fan_in, fan_out))
            if fan_out != layer_sizes[-1]:      # hidden layers only
                if use_batchnorm:
                    layers.append(nn.BatchNorm1d(fan_out))
                layers.append(act_fn())
                if dropout_rate > 0.0:
                    layers.append(nn.Dropout(dropout_rate))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Forward pass: flatten then apply MLP.

        Args:
            x: Image tensor of shape (N, 1, 28, 28) or (N, 784).

        Returns:
            Class logits of shape (N, num_classes).
        '''
        return self.net(x.view(x.size(0), -1))


def apply_he_init(model: nn.Module) -> None:
    '''Apply Kaiming-normal initialisation to all Linear layers.

    Uses nonlinearity="relu" so the gain matches ReLU's variance halving.
    See 05-08 for derivation.

    Args:
        model: Module whose Linear layers will be initialised in-place.
    '''
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)


def count_parameters(model: nn.Module) -> tuple[int, int]:
    '''Count total and trainable parameters.

    Args:
        model: PyTorch module.

    Returns:
        Tuple of (total_params, trainable_params).
    '''
    total      = sum(p.numel() for p in model.parameters())
    trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def train_one_epoch(
    model:      nn.Module,
    dataloader: DataLoader,
    optimizer:  optim.Optimizer,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Train for one epoch and return (avg_loss, accuracy).

    Args:
        model:      Network to train.
        dataloader: Training DataLoader.
        optimizer:  Optimizer instance.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in dataloader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out  = model(X_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_b.size(0)
        correct      += out.argmax(1).eq(y_b).sum().item()
        total        += X_b.size(0)
    return running_loss / total, correct / total


def evaluate(
    model:      nn.Module,
    dataloader: DataLoader,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Evaluate model and return (avg_loss, accuracy).

    Args:
        model:      Network to evaluate.
        dataloader: DataLoader for the evaluation split.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_b, y_b in dataloader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            out  = model(X_b)
            loss = criterion(out, y_b)
            running_loss += loss.item() * X_b.size(0)
            correct      += out.argmax(1).eq(y_b).sum().item()
            total        += X_b.size(0)
    return running_loss / total, correct / total


print("FlexMLP, apply_he_init, count_parameters, train_one_epoch, evaluate — defined.")


In [ ]:
# ── Architecture search: 3-epoch trial on 10 K subset ─────────────────────────
# Build a fast search DataLoader on a random 10K subset of the training set.
rng_search   = np.random.default_rng(SEED)
search_idx   = rng_search.choice(len(train_set_f), SEARCH_TRAIN_N, replace=False)
search_set   = Subset(train_set_f, search_idx.tolist())
search_loader = DataLoader(search_set, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=0, pin_memory=torch.cuda.is_available())

# Architecture candidates: (name, hidden_sizes)
arch_candidates: list[tuple[str, list[int]]] = [
    ("Tiny       [128]",           [INPUT_DIM, 128,           NUM_CLASSES]),
    ("Small      [256,128]",       [INPUT_DIM, 256, 128,      NUM_CLASSES]),
    ("Medium     [512,256]",       [INPUT_DIM, 512, 256,      NUM_CLASSES]),
    ("Deep-3     [256,256,256]",   [INPUT_DIM, 256, 256, 256, NUM_CLASSES]),
    ("Deep-wide  [512,256,128]",   [INPUT_DIM, 512, 256, 128, NUM_CLASSES]),
    ("XL         [1024,512,256]",  [INPUT_DIM, 1024, 512, 256, NUM_CLASSES]),
]

criterion_s  = nn.CrossEntropyLoss()
search_rows: list[dict] = []

print(f"Architecture search: {SEARCH_EPOCHS} epochs on {SEARCH_TRAIN_N:,} samples")
print(f"  {'Architecture':<32s}  {'Params':>8s}  {'Val Acc':>7s}  {'Time(s)':>7s}")
print("  " + "-" * 60)

for arch_name, sizes in arch_candidates:
    torch.manual_seed(SEED)
    m_s = FlexMLP(sizes, dropout_rate=DROPOUT_RATE).to(device)
    apply_he_init(m_s)
    total_p, _ = count_parameters(m_s)
    o_s = optim.AdamW(m_s.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    t0_s = time.time()
    for _ in range(SEARCH_EPOCHS):
        train_one_epoch(m_s, search_loader, o_s, criterion_s, device)
    _, val_acc_s = evaluate(m_s, val_loader, criterion_s, device)
    elapsed_s = time.time() - t0_s
    search_rows.append({
        "Architecture": arch_name,
        "Params":       total_p,
        "Val Acc":      val_acc_s,
        "Time (s)":     elapsed_s,
    })
    print(f"  {arch_name:<32s}  {total_p:>8,}  {val_acc_s:>7.2%}  {elapsed_s:>7.1f}")


In [ ]:
# ── Visualise search results and select best architecture ─────────────────────
search_df = pd.DataFrame(search_rows).sort_values("Val Acc", ascending=False)
print("\nSearch results (sorted by Val Acc):")
print(search_df[["Architecture", "Params", "Val Acc", "Time (s)"]].to_string(index=False))

fig_sr, axes_sr = plt.subplots(1, 2, figsize=(13, 4))
arch_labels = [r["Architecture"].split("[")[0].strip() for r in search_rows]
val_accs_sr = [r["Val Acc"] for r in search_rows]
params_sr   = [r["Params"]  for r in search_rows]
x_sr        = list(range(len(search_rows)))
bar_cols_sr = plt.cm.viridis(np.linspace(0.2, 0.9, len(search_rows)))

for ax_sr, y_sr, ylabel_sr, title_sr in zip(
    axes_sr,
    [val_accs_sr, params_sr],
    ["Val Accuracy (3 epochs)", "Total Parameters"],
    ["Architecture Search: Val Accuracy", "Architecture Search: Parameter Count"],
):
    bars_sr = ax_sr.bar(x_sr, y_sr, color=bar_cols_sr, edgecolor="k", lw=0.6)
    for b_sr, v_sr in zip(bars_sr, y_sr):
        label_sr = f"{v_sr:.1%}" if isinstance(v_sr, float) else f"{v_sr:,}"
        ax_sr.text(b_sr.get_x() + b_sr.get_width() / 2.0,
                   b_sr.get_height() * 1.01, label_sr,
                   ha="center", va="bottom", fontsize=7)
    ax_sr.set_xticks(x_sr)
    ax_sr.set_xticklabels(arch_labels, rotation=30, ha="right", fontsize=8)
    ax_sr.set_ylabel(ylabel_sr, fontsize=11)
    ax_sr.set_title(title_sr, fontsize=11, fontweight="bold")
    ax_sr.grid(axis="y", alpha=0.3)

plt.suptitle("Architecture Search Results (3-epoch proxy)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Select best architecture by val accuracy
best_row     = max(search_rows, key=lambda r: r["Val Acc"])
best_arch_nm = best_row["Architecture"]
best_sizes   = next(s for n, s in arch_candidates if n == best_arch_nm)
print(f"\nBest architecture: {best_arch_nm}")
print(f"  Sizes:    {best_sizes}")
print(f"  Val acc:  {best_row['Val Acc']:.2%}")
print(f"  Params:   {best_row['Params']:,}")


### 1.3 Regularisation Ablation

Using the **best architecture** from the search, we isolate the contribution of each regularisation technique: **(a)** no regularisation, **(b)** dropout only, **(c)** dropout + BatchNorm.  All runs use the same 5-epoch proxy training on `SEARCH_TRAIN_N` samples to keep comparisons fair.


In [ ]:
# ── Regularisation Ablation: Dropout vs BatchNorm vs Both ────────────────────
# Using the best architecture from the search, we run a quick 5-epoch proxy
# training on SEARCH_TRAIN_N samples with three regularisation configs:
#   (a) No regularisation  (dropout=0, batchnorm=False)
#   (b) Dropout only       (dropout=DROPOUT_RATE, batchnorm=False)
#   (c) Dropout + BN       (dropout=DROPOUT_RATE, batchnorm=True)
# This isolates the contribution of each technique.

REG_PROXY_EPOCHS = 5

reg_configs: list[dict] = [
    {"label": "No regularisation",  "dropout": 0.0,          "batchnorm": False},
    {"label": "Dropout only",        "dropout": DROPOUT_RATE, "batchnorm": False},
    {"label": "Dropout + BatchNorm", "dropout": DROPOUT_RATE, "batchnorm": True},
]

# Build proxy datasets (same as arch search)
rng_reg   = np.random.default_rng(SEED + 99)
reg_idx   = rng_reg.choice(len(train_full), size=SEARCH_TRAIN_N, replace=False)
reg_sub   = torch.utils.data.Subset(train_full, reg_idx.tolist())
val_full_ = torch.utils.data.Subset(train_full,
            list(set(range(len(train_full))) - set(reg_idx.tolist())))
# keep val set small for speed
val_reg_idx = rng_reg.choice(len(val_full_), size=2000, replace=False)
val_reg_sub = torch.utils.data.Subset(val_full_, val_reg_idx.tolist())

reg_train_loader = torch.utils.data.DataLoader(
    reg_sub, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
reg_val_loader = torch.utils.data.DataLoader(
    val_reg_sub, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)

reg_criterion = nn.CrossEntropyLoss()
reg_results: list[dict] = []

for cfg in reg_configs:
    reg_model = FlexMLP(
        best_arch_dims,
        dropout_rate=cfg["dropout"],
        use_batchnorm=cfg["batchnorm"],
    ).to(device)
    apply_he_init(reg_model)
    reg_opt   = torch.optim.AdamW(reg_model.parameters(),
                                   lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    reg_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        reg_opt, T_max=REG_PROXY_EPOCHS, eta_min=ETA_MIN
    )

    ep_val_accs: list[float] = []
    for ep in range(REG_PROXY_EPOCHS):
        train_one_epoch(reg_model, reg_train_loader, reg_criterion, reg_opt, device)
        _, v_acc = evaluate(reg_model, reg_val_loader, reg_criterion, device)
        ep_val_accs.append(v_acc)
        reg_sched.step()

    total_p, trainable_p = count_parameters(reg_model)
    reg_results.append({
        "Config":       cfg["label"],
        "Final Val Acc": ep_val_accs[-1],
        "Curve":        ep_val_accs,
        "Params":       trainable_p,
    })
    print(f"  {cfg['label']:28s}  val_acc={ep_val_accs[-1]:.4f}")

# Print table
print("\nRegularisation Ablation Results:")
print(f"  {'Config':<28s}  {'Val Acc':>8s}  {'Delta':>8s}")
base_acc = reg_results[0]["Final Val Acc"]
for row in reg_results:
    delta = row["Final Val Acc"] - base_acc
    delta_str = f"{delta:+.4f}" if delta != 0.0 else "  baseline"
    print(f"  {row['Config']:<28s}  {row['Final Val Acc']:>8.4f}  {delta_str:>8s}")

# Learning curve comparison
fig_reg, ax_reg = plt.subplots(figsize=(8, 4))
reg_colors = ["#d62728", "#1f77b4", "#2ca02c"]
for row, col in zip(reg_results, reg_colors):
    ax_reg.plot(range(1, REG_PROXY_EPOCHS + 1), row["Curve"],
                marker="o", color=col, lw=2, label=row["Config"])
ax_reg.set_xlabel("Epoch", fontsize=11)
ax_reg.set_ylabel("Val Accuracy", fontsize=11)
ax_reg.set_title(f"Regularisation Ablation — {REG_PROXY_EPOCHS}-Epoch Proxy Training",
                  fontsize=11, fontweight="bold")
ax_reg.legend(fontsize=9)
ax_reg.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Part 2 — Full Training Pipeline

With the best architecture chosen, we train the complete model for up to
{NUM_EPOCHS} epochs using:

- **He (Kaiming) normal** initialisation — proven optimal for ReLU at depth (05-08)
- **AdamW** with weight decay — prevents weight explosion without coupling decay
  to adaptive scaling (05-09)
- **Cosine annealing** LR schedule — decays LR smoothly to `ETA_MIN`
- **Dropout(0.3)** — reduces co-adaptation between neurons (05-05)
- **Early stopping** (patience = {PATIENCE}) — halts training when val loss stalls


### Part 2 — Full Training

Using the best architecture from the search, we now train with full data, He initialisation, AdamW optimiser, cosine annealing learning-rate schedule, and early stopping. This combines techniques from 05-08 (init), 05-09 (optimisers), and 05-05 (regularisation).

In [ ]:
# ── Build full model with best architecture + He init ─────────────────────────
torch.manual_seed(SEED)
model = FlexMLP(best_sizes, dropout_rate=DROPOUT_RATE).to(device)
apply_he_init(model)  # He (Kaiming) — see 05-08 for derivation

total_p, trainable_p = count_parameters(model)
print(f"Model architecture: {best_sizes}")
print(f"Total params:     {total_p:,}")
print(f"Trainable params: {trainable_p:,}")

# Per-layer parameter breakdown
print("\nLayer-by-layer breakdown:")
print(f"  {'Layer':<35s}  {'Shape':>20s}  {'Params':>10s}")
print("  " + "-" * 68)
for name, param in model.named_parameters():
    print(f"  {name:<35s}  {str(list(param.shape)):>20s}  {param.numel():>10,}")

# Sanity check: forward pass
dummy_batch = torch.randn(4, 1, 28, 28).to(device)
with torch.no_grad():
    dummy_out = model(dummy_batch)
print(f"\nSanity check: input {list(dummy_batch.shape)} → output {list(dummy_out.shape)}")
assert dummy_out.shape == (4, NUM_CLASSES), "Output shape mismatch!"
print("Output shape correct.")


In [ ]:
# ── Full training with AdamW + CosineAnnealingLR + early stopping ─────────────
optimizer_full  = optim.AdamW(model.parameters(),
                               lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler_full  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_full, T_max=NUM_EPOCHS, eta_min=ETA_MIN,
)
criterion_full  = nn.CrossEntropyLoss()

train_losses: list[float] = []
val_losses:   list[float] = []
train_accs:   list[float] = []
val_accs:     list[float] = []
lr_history:   list[float] = []

best_val_loss    = float("inf")
best_model_state: dict = {}
patience_counter = 0
stopped_epoch    = NUM_EPOCHS

t0_full = time.time()

for epoch in range(NUM_EPOCHS):
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, optimizer_full, criterion_full, device
    )
    vl_loss, vl_acc = evaluate(model, val_loader, criterion_full, device)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)
    lr_history.append(scheduler_full.get_last_lr()[0])

    elapsed_full = time.time() - t0_full
    print(f"Epoch {epoch+1:>2d}/{NUM_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f} | Val Loss: {vl_loss:.4f} | "
          f"Val Acc: {vl_acc:.2%} | Time: {elapsed_full:.1f}s")

    # Best model tracking
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= PATIENCE:
        stopped_epoch = epoch + 1
        print(f"  Early stopping triggered at epoch {stopped_epoch}.")
        break

    scheduler_full.step()

# Restore best model weights
model.load_state_dict(best_model_state)
total_time = time.time() - t0_full
print(f"\nTraining complete: {stopped_epoch} epoch(s), {total_time:.1f}s total.")
print(f"Best val loss: {best_val_loss:.4f}")


In [ ]:
# ── Training curves + LR schedule ─────────────────────────────────────────────
n_trained    = len(train_losses)
epochs_ax    = list(range(1, n_trained + 1))

fig_tc, axes_tc = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
axes_tc[0].plot(epochs_ax, train_losses, lw=2, color="#1f77b4", label="Train")
axes_tc[0].plot(epochs_ax, val_losses,   lw=2, color="#ff7f0e", label="Val",
                linestyle="--")
axes_tc[0].set_xlabel("Epoch", fontsize=11)
axes_tc[0].set_ylabel("Cross-Entropy Loss", fontsize=11)
axes_tc[0].set_title("Loss Curves", fontsize=11, fontweight="bold")
axes_tc[0].legend(fontsize=9)
axes_tc[0].grid(alpha=0.3)

# Accuracy curves
axes_tc[1].plot(epochs_ax, [a * 100 for a in train_accs], lw=2,
                color="#1f77b4", label="Train")
axes_tc[1].plot(epochs_ax, [a * 100 for a in val_accs],   lw=2,
                color="#ff7f0e", label="Val", linestyle="--")
axes_tc[1].set_xlabel("Epoch", fontsize=11)
axes_tc[1].set_ylabel("Accuracy (%)", fontsize=11)
axes_tc[1].set_title("Accuracy Curves", fontsize=11, fontweight="bold")
axes_tc[1].legend(fontsize=9)
axes_tc[1].grid(alpha=0.3)
axes_tc[1].set_ylim(50, 101)

# LR schedule
axes_tc[2].plot(epochs_ax, lr_history, lw=2, color="#2ca02c")
axes_tc[2].set_xlabel("Epoch", fontsize=11)
axes_tc[2].set_ylabel("Learning Rate", fontsize=11)
axes_tc[2].set_title("Cosine Annealing LR Schedule", fontsize=11, fontweight="bold")
axes_tc[2].grid(alpha=0.3)
axes_tc[2].set_yscale("log")

plt.suptitle("Full Training: Loss, Accuracy, and LR Schedule",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Gap check: large gap → overfitting, small gap → underfitting
final_tr_acc = train_accs[-1]
final_vl_acc = val_accs[-1]
gap          = final_tr_acc - final_vl_acc
print(f"Final train acc: {final_tr_acc:.2%}  val acc: {final_vl_acc:.2%}  "
      f"gap: {gap:.2%}")
status = ("well-fit" if gap < 0.03
          else "mildly overfit" if gap < 0.07
          else "overfit — consider more dropout or less capacity")
print(f"Diagnosis: {status}")


### 2.3 Learning-Rate Schedule Visualisation

We use **Cosine Annealing** (`CosineAnnealingLR`) to decay the learning rate smoothly from `LEARNING_RATE` to `ETA_MIN` over `NUM_EPOCHS` steps.  Plotting the schedule alongside the loss curves helps verify that the model converges during the low-LR fine-tuning phase rather than stagnating early.


In [ ]:
# ── Learning-Rate Schedule: Cosine Annealing Visualisation ───────────────────
# CosineAnnealingLR decays the learning rate from lr_max to eta_min following
# a cosine curve over T_max epochs.  Plotting it helps confirm the schedule
# is set up correctly and understand when the model is being fine-tuned vs
# exploring (high LR phase).

# Reconstruct the theoretical LR trajectory for NUM_EPOCHS steps

dummy_param   = torch.nn.Parameter(torch.zeros(1))
dummy_opt     = torch.optim.AdamW([dummy_param], lr=LEARNING_RATE)
dummy_sched   = torch.optim.lr_scheduler.CosineAnnealingLR(
    dummy_opt, T_max=NUM_EPOCHS, eta_min=ETA_MIN
)
lr_trajectory: list[float] = []
for _ in range(NUM_EPOCHS):
    lr_trajectory.append(dummy_opt.param_groups[0]["lr"])
    dummy_sched.step()

# Overlay actual history (if shorter, due to early stopping)
epochs_ran = list(range(1, len(train_losses) + 1))

fig_lr, axes_lr = plt.subplots(1, 2, figsize=(12, 4))

# Left: LR curve
ax_lr = axes_lr[0]
ax_lr.plot(range(1, NUM_EPOCHS + 1), lr_trajectory, color="#1f77b4",
           lw=2, label="Theoretical LR")
ax_lr.axvline(len(train_losses), color="crimson", ls="--", lw=1.5,
              label=f"Early-stop epoch {len(train_losses)}")
ax_lr.set_xlabel("Epoch", fontsize=11)
ax_lr.set_ylabel("Learning Rate", fontsize=11)
ax_lr.set_title("Cosine Annealing LR Schedule", fontsize=11, fontweight="bold")
ax_lr.legend(fontsize=9)
ax_lr.set_yscale("log")
ax_lr.grid(True, alpha=0.3)

# Right: Train loss vs LR to see loss-drop correlation
ax_ll = axes_lr[1]
color_loss = "#2ca02c"
color_lr   = "#ff7f0e"
ax2        = ax_ll.twinx()
ax_ll.plot(epochs_ran, train_losses, color=color_loss, lw=2, label="Train Loss")
ax_ll.plot(epochs_ran, val_losses,   color=color_loss, lw=2, ls="--",
           alpha=0.6, label="Val Loss")
ax2.plot(epochs_ran, lr_trajectory[:len(train_losses)], color=color_lr,
         lw=1.5, ls=":", label="LR (right axis)")
ax_ll.set_xlabel("Epoch", fontsize=11)
ax_ll.set_ylabel("Loss", fontsize=11, color=color_loss)
ax2.set_ylabel("Learning Rate", fontsize=11, color=color_lr)
ax_ll.set_title("Train/Val Loss vs LR", fontsize=11, fontweight="bold")
lines1, labels1 = ax_ll.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax_ll.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")
ax_ll.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cosine formula reminder
lr_epoch0     = LEARNING_RATE
lr_epochT     = ETA_MIN + 0.5 * (LEARNING_RATE - ETA_MIN) * (1 + math.cos(math.pi))
print("LR schedule summary:")
print(f"  Start (epoch 1): {lr_trajectory[0]:.2e}")
print(f"  End   (epoch {NUM_EPOCHS}): {lr_trajectory[-1]:.2e}")
print(f"  Theoretical final (cosine formula): {lr_epochT:.2e}")
print(f"  Early-stop fired at epoch: {len(train_losses)}")
print(f"  LR at stop: {lr_trajectory[len(train_losses)-1]:.2e}")


---
## Part 3 — Comprehensive Evaluation

A single accuracy number on the test set is rarely enough.  We report:

1. **Overall accuracy** and test loss.
2. **Confusion matrix** — which classes are confused with each other?
3. **Per-class metrics** — precision, recall, F1 (balanced classes → macro average).
4. **Error analysis** — visually inspect misclassified examples.


### Part 3 — Evaluation

We evaluate the trained model on the held-out test set using accuracy, per-class precision/recall/F1, confusion matrix analysis, and visual error inspection. This mirrors the evaluation methodology from Module 04.

In [ ]:
# ── Test set evaluation ────────────────────────────────────────────────────────
test_loss, test_acc = evaluate(model, test_loader, criterion_full, device)
print(f"Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.2%}")

# Collect all predictions and labels
all_preds:  list[int] = []
all_labels: list[int] = []
all_probs:  list[np.ndarray] = []

model.eval()
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        logits = model(X_b)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(y_b.numpy().tolist())
        all_probs.extend(probs.tolist())

all_preds_np  = np.array(all_preds)
all_labels_np = np.array(all_labels)
all_probs_np  = np.array(all_probs)

print(f"\nTotal test samples: {len(all_preds_np):,}")
print(f"Correctly classified: {(all_preds_np == all_labels_np).sum():,}")
print(f"Misclassified: {(all_preds_np != all_labels_np).sum():,}")

# Confidence statistics
correct_mask  = all_preds_np == all_labels_np
max_probs     = all_probs_np.max(axis=1)
print(f"\nMean confidence (correct):   {max_probs[correct_mask].mean():.3f}")
print(f"Mean confidence (incorrect): {max_probs[~correct_mask].mean():.3f}")
print("Lower confidence on incorrect predictions confirms the model is somewhat calibrated.")


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels_np, all_preds_np)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # normalised by row

fig_cm, axes_cm = plt.subplots(1, 2, figsize=(16, 6))
short_labels = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
                "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]

for ax_cm, data_cm, title_cm, fmt_cm in zip(
    axes_cm,
    [cm,      cm_norm],
    ["Counts", "Row-normalised (Recall)"],
    [True,     False],
):
    im_cm = ax_cm.imshow(data_cm, cmap="Blues", vmin=0)
    ax_cm.set_xticks(range(NUM_CLASSES))
    ax_cm.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
    ax_cm.set_yticks(range(NUM_CLASSES))
    ax_cm.set_yticklabels(short_labels, fontsize=8)
    ax_cm.set_xlabel("Predicted", fontsize=11)
    ax_cm.set_ylabel("True", fontsize=11)
    ax_cm.set_title(f"Confusion Matrix — {title_cm}", fontsize=11, fontweight="bold")
    plt.colorbar(im_cm, ax=ax_cm, shrink=0.85)

    thresh_cm = data_cm.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            val_cm = data_cm[i, j]
            text_cm = f"{val_cm:.0f}" if fmt_cm else f"{val_cm:.2f}"
            ax_cm.text(j, i, text_cm, ha="center", va="center", fontsize=6,
                       color="white" if val_cm > thresh_cm else "black")

plt.suptitle("Confusion Matrix — FashionMNIST Test Set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Most common confusions
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)
top_confusions_flat = np.argsort(cm_offdiag.ravel())[::-1][:5]
print("Top 5 most common misclassifications:")
for flat_idx in top_confusions_flat:
    true_cls  = flat_idx // NUM_CLASSES
    pred_cls  = flat_idx  % NUM_CLASSES
    cnt_cf    = cm[true_cls, pred_cls]
    print(f"  True: {FASHION_CLASSES[true_cls]:<15s} → Pred: {FASHION_CLASSES[pred_cls]:<15s}  "
          f"({cnt_cf} times)")


In [ ]:
# ── Per-class precision, recall, F1 ──────────────────────────────────────────
report_str = classification_report(
    all_labels_np, all_preds_np,
    target_names=FASHION_CLASSES, digits=3,
)
print("Classification Report:")
print(report_str)

precision_cls = precision_score(all_labels_np, all_preds_np, average=None)
recall_cls    = recall_score(   all_labels_np, all_preds_np, average=None)
f1_cls        = f1_score(       all_labels_np, all_preds_np, average=None)

# Grouped bar chart: precision / recall / F1 per class
x_pc   = np.arange(NUM_CLASSES)
width  = 0.28
fig_pc, ax_pc = plt.subplots(figsize=(13, 5))
ax_pc.bar(x_pc - width, precision_cls, width, label="Precision", color="#1f77b4", alpha=0.85)
ax_pc.bar(x_pc,         recall_cls,    width, label="Recall",    color="#ff7f0e", alpha=0.85)
ax_pc.bar(x_pc + width, f1_cls,        width, label="F1",        color="#2ca02c", alpha=0.85)
ax_pc.set_xticks(x_pc)
ax_pc.set_xticklabels(FASHION_CLASSES, rotation=30, ha="right", fontsize=9)
ax_pc.set_ylabel("Score", fontsize=11)
ax_pc.set_ylim(0, 1.08)
ax_pc.set_title("Per-Class Precision, Recall, F1 — FashionMNIST Test Set",
                fontsize=11, fontweight="bold")
ax_pc.legend(fontsize=9, loc="lower right")
ax_pc.axhline(f1_score(all_labels_np, all_preds_np, average="macro"),
              color="k", lw=1, ls="--", alpha=0.5, label="Macro F1")
ax_pc.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

macro_f1 = f1_score(all_labels_np, all_preds_np, average="macro")
worst_cls = FASHION_CLASSES[int(np.argmin(f1_cls))]
best_cls  = FASHION_CLASSES[int(np.argmax(f1_cls))]
print(f"Macro F1: {macro_f1:.4f}")
print(f"Hardest class: {worst_cls} (F1={f1_cls.min():.3f})")
print(f"Easiest class: {best_cls}  (F1={f1_cls.max():.3f})")


In [ ]:
# ── Error analysis: visualise misclassified examples ─────────────────────────
# Collect actual image tensors for the test set
test_images_list: list[torch.Tensor] = []
test_labels_list: list[int]          = []
for img_e, lbl_e in test_set_f:
    test_images_list.append(img_e)
    test_labels_list.append(int(lbl_e))

test_images_np  = np.stack([t.numpy() for t in test_images_list])   # (10000, 1, 28, 28)
test_labels_arr = np.array(test_labels_list)

# Find misclassified indices
wrong_idx = np.where(all_preds_np != all_labels_np)[0]
print(f"Total misclassified: {len(wrong_idx)} / {len(all_labels_np)} "
      f"({len(wrong_idx) / len(all_labels_np):.1%})")

# Show a 5×5 grid of errors with true / predicted label + confidence
n_show_err = 25
rng_err    = np.random.default_rng(SEED + 42)
show_idx   = rng_err.choice(wrong_idx, size=min(n_show_err, len(wrong_idx)),
                              replace=False)

fig_err, axes_err = plt.subplots(5, 5, figsize=(12, 12))
for ax_e, idx_e in zip(axes_err.ravel(), show_idx):
    img_e   = test_images_np[idx_e, 0]     # (28, 28)
    true_e  = all_labels_np[idx_e]
    pred_e  = all_preds_np[idx_e]
    conf_e  = all_probs_np[idx_e, pred_e]
    ax_e.imshow(img_e, cmap="gray")
    ax_e.axis("off")
    ax_e.set_title(
        f"T: {FASHION_CLASSES[true_e][:8]}\n"
        f"P: {FASHION_CLASSES[pred_e][:8]} ({conf_e:.0%})",
        fontsize=6.5, color="red",
    )

plt.suptitle("Misclassified Examples (True → Predicted, confidence)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Per-class error rate
print("\nPer-class error rate (sorted by error rate descending):")
per_class_err: list[tuple[str, float, int]] = []
for cls_i in range(NUM_CLASSES):
    cls_mask   = test_labels_arr == cls_i
    err_rate   = float((all_preds_np[cls_mask] != cls_i).mean())
    n_wrong    = int((all_preds_np[cls_mask] != cls_i).sum())
    per_class_err.append((FASHION_CLASSES[cls_i], err_rate, n_wrong))

per_class_err.sort(key=lambda x: x[1], reverse=True)
for cls_name, err_r, n_wrong_c in per_class_err:
    print(f"  {cls_name:<15s}  error rate: {err_r:.2%}  ({n_wrong_c} errors)")


---
## Part 4 — ONNX Export & Inference Profiling

**ONNX (Open Neural Network Exchange)** is a cross-framework model format.
Exporting a trained PyTorch model to ONNX enables deployment in runtimes such as
ONNX Runtime, TensorRT, and OpenCV DNN — often with significant speed gains over
the Python/PyTorch interpreter.

We also profile the model to understand its computational footprint.


### Part 4 — Export & Deployment

With a trained model in hand, we export to ONNX format for portable inference and profile single-batch latency. The ONNX export uses  (built into PyTorch); the optional verification steps require  and  packages.

In [ ]:
# ── ONNX export ───────────────────────────────────────────────────────────────
# Move model to CPU for export (portable, works anywhere)
model_cpu = model.cpu().eval()
dummy_in  = torch.randn(1, 1, 28, 28)    # one FashionMNIST image

pathlib.Path(ONNX_PATH).parent.mkdir(parents=True, exist_ok=True)

torch.onnx.export(
    model_cpu,
    dummy_in,
    ONNX_PATH,
    opset_version=11,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={
        "image":  {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    verbose=False,
)

onnx_size_kb = pathlib.Path(ONNX_PATH).stat().st_size / 1024
print(f"ONNX model exported to: {ONNX_PATH}")
print(f"File size: {onnx_size_kb:.1f} KB")

# ── Verify ONNX model (requires onnx package — optional) ──────────────────────
try:
    import onnx
    onnx_model = onnx.load(ONNX_PATH)
    onnx.checker.check_model(onnx_model)
    print(f"ONNX check: PASS  (opset version {onnx_model.opset_import[0].version})")
    print(f"Graph inputs:  {[inp.name for inp in onnx_model.graph.input]}")
    print(f"Graph outputs: {[out.name for out in onnx_model.graph.output]}")
    n_nodes = len(onnx_model.graph.node)
    print(f"Graph nodes (ops): {n_nodes}")
except ImportError:
    print("onnx package not installed; skipping model graph check.")
    print("Install with: pip install onnx")

# ── Cross-check: ONNX Runtime inference matches PyTorch ───────────────────────
try:
    import onnxruntime as ort
    ort_session = ort.InferenceSession(ONNX_PATH)
    test_img_np = dummy_in.numpy()
    ort_out     = ort_session.run(["logits"], {"image": test_img_np})[0]
    with torch.no_grad():
        pt_out = model_cpu(dummy_in).numpy()
    max_diff = float(np.abs(ort_out - pt_out).max())
    print(f"\nONNX Runtime vs PyTorch max absolute difference: {max_diff:.2e}")
    print(f"Match: {'YES' if max_diff < 1e-4 else 'NO (check opset)'}")
except ImportError:
    print("\tonnxruntime not installed; skipping cross-check.")
    print("\tInstall with: pip install onnxruntime")

# Move model back to device for profiling
model.to(device)


In [ ]:
# ── Inference profiling ────────────────────────────────────────────────────────
total_p, trainable_p = count_parameters(model)
print(f"Model parameter count:")
print(f"  Total:     {total_p:,}")
print(f"  Trainable: {trainable_p:,}")
print(f"  Memory (fp32): {total_p * 4 / 1024:.1f} KB  "
      f"({total_p * 4 / 1024 / 1024:.2f} MB)")

# ── FLOPs approximation (forward pass only) ───────────────────────────────────
# For a linear layer with fan_in, fan_out: FLOPs ≈ 2 * fan_in * fan_out
flops_fwd = 0
for module in model.modules():
    if isinstance(module, nn.Linear):
        fan_in_f  = module.weight.shape[1]
        fan_out_f = module.weight.shape[0]
        flops_fwd += 2 * fan_in_f * fan_out_f   # multiply-accumulate ops
print(f"Estimated FLOPs (1 forward pass): {flops_fwd:,}  ({flops_fwd / 1e6:.2f} MFLOPs)")

# ── Batch inference speed ─────────────────────────────────────────────────────
N_WARMUP   = 5
N_TIMING   = 50
batch_test = torch.randn(256, 1, 28, 28).to(device)

model.eval()
with torch.no_grad():
    for _ in range(N_WARMUP):
        _ = model(batch_test)

    t_start = time.time()
    for _ in range(N_TIMING):
        _ = model(batch_test)
    t_end = time.time()

total_samples = N_TIMING * 256
elapsed_inf   = t_end - t_start
throughput    = total_samples / elapsed_inf
latency_ms    = elapsed_inf / N_TIMING * 1000

print(f"\nBatch inference (batch=256, {N_TIMING} runs on {device}):")
print(f"  Throughput: {throughput:,.0f} samples/second")
print(f"  Latency:    {latency_ms:.2f} ms/batch  ({latency_ms/256*1000:.3f} μs/sample)")

# ── Summary: parameter efficiency ─────────────────────────────────────────────
print(f"\nParameter efficiency: {test_acc:.4f} test acc / {total_p/1e6:.3f}M params "
      f"= {test_acc / (total_p / 1e6):.3f} acc per 1M params")


# ── Batch-size throughput sweep ───────────────────────────────────────────────
# Throughput (samples/sec) typically increases with batch size until GPU/CPU
# memory bandwidth is saturated.  This sweep shows the trade-off on the
# current hardware.
batch_sizes_sweep: list[int] = [1, 8, 32, 128, 256, 512]
throughput_sweep:  list[float] = []
latency_sweep:     list[float] = []

model_cpu_prof = best_model.cpu().eval()
dummy_inputs   = {bs: torch.randn(bs, INPUT_DIM) for bs in batch_sizes_sweep}

WARMUP_REPS = 5
TIME_REPS    = 20

for bs in batch_sizes_sweep:
    inp = dummy_inputs[bs]
    # Warm-up
    with torch.no_grad():
        for _ in range(WARMUP_REPS):
            _ = model_cpu_prof(inp)
    # Timed
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(TIME_REPS):
            _ = model_cpu_prof(inp)
    elapsed  = time.perf_counter() - t0
    lat_ms   = elapsed / TIME_REPS * 1_000          # ms per batch
    tput     = bs * TIME_REPS / elapsed              # samples / sec
    throughput_sweep.append(tput)
    latency_sweep.append(lat_ms)

best_model.to(device)   # restore to original device

print("\nBatch-Size Throughput Sweep (CPU inference):")
print(f"  {'Batch':>6s}  {'Throughput (samp/s)':>22s}  {'Latency (ms/batch)':>20s}")
for bs, tput, lat in zip(batch_sizes_sweep, throughput_sweep, latency_sweep):
    print(f"  {bs:>6d}  {tput:>22.1f}  {lat:>20.2f}")

fig_tput, ax_tput = plt.subplots(figsize=(7, 3.5))
ax_tput.plot(batch_sizes_sweep, throughput_sweep, marker="o", color="#1f77b4", lw=2)
ax_tput.set_xscale("log", base=2)
ax_tput.set_xlabel("Batch Size", fontsize=11)
ax_tput.set_ylabel("Throughput (samples / sec)", fontsize=11)
ax_tput.set_title("Inference Throughput vs Batch Size (CPU)", fontsize=11, fontweight="bold")
ax_tput.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 5 — Results & Comparison

We compare our optimised pipeline against approximate baselines from earlier notebooks in this module. The pipeline integrates architecture search, He initialisation, AdamW, cosine annealing, and early stopping — the combination of best practices should exceed any single-technique baseline.

In [ ]:
# ── Final comparison: our pipeline vs baselines ───────────────────────────────
# Approximate baselines from prior notebooks (reference — not re-run here).
# 05-02: basic MLP 784→256→128→10, no regularisation, Adam, ~87-88% test acc
# 05-05: with Dropout + L1/L2, modest improvement ~88-89%
# 05-08: He init on deep MLP (MNIST, ~99%+; FashionMNIST reference ~88%)
# 05-09: AdamW on FashionMNIST in 10 epochs → varies ~87-89%

baseline_rows = [
    {
        "Pipeline":       "Random Baseline",
        "Init":           "—",
        "Optimiser":      "—",
        "Regularisation": "—",
        "Epochs":         "—",
        "Test Acc":       "10.00%",
        "Notes":          "10 classes, uniform guess",
    },
    {
        "Pipeline":       "05-02 Basic MLP",
        "Init":           "PyTorch default",
        "Optimiser":      "Adam",
        "Regularisation": "None",
        "Epochs":         "10",
        "Test Acc":       "~87.5%",
        "Notes":          "784→256→128→10",
    },
    {
        "Pipeline":       "05-05 + Dropout/L2",
        "Init":           "PyTorch default",
        "Optimiser":      "Adam",
        "Regularisation": "Dropout + L2",
        "Epochs":         "10",
        "Test Acc":       "~88.5%",
        "Notes":          "Regularisation reduces gap",
    },
    {
        "Pipeline":       "05-09 AdamW (10 ep)",
        "Init":           "He normal",
        "Optimiser":      "AdamW",
        "Regularisation": "Dropout(0.3)",
        "Epochs":         "10",
        "Test Acc":       "~89.0%",
        "Notes":          "Best 10-epoch result",
    },
    {
        "Pipeline":       "05-10 Full Pipeline ← THIS",
        "Init":           "He normal",
        "Optimiser":      "AdamW + cosine LR",
        "Regularisation": "Dropout(0.3) + WD",
        "Epochs":         str(stopped_epoch),
        "Test Acc":       f"{test_acc:.2%}",
        "Notes":          "Architecture search + early stop",
    },
]

final_df = pd.DataFrame(baseline_rows)
print("End-to-End Pipeline Comparison:")
print(final_df.to_string(index=False))

# Highlight improvement
print(f"\nOur pipeline achieves {test_acc:.2%} test accuracy.")
print(f"  vs random baseline: +{(test_acc - 0.10)*100:.1f} pp")
print(f"  vs naive MLP:       +{(test_acc - 0.875)*100:.1f} pp (approx.)")
print(f"Remaining error: {(1 - test_acc)*100:.1f}% — primarily from 'Shirt' vs "
      f"'T-shirt/Pullover' ambiguity (confirmed by confusion matrix).")

# ── Techniques inventory ───────────────────────────────────────────────────────
techniques = [
    ("Data normalisation",           "transforms.Normalize",          "Part 0"),
    ("Architecture search",          "3-epoch proxy trial",           "Part 1"),
    ("He (Kaiming) normal init",     "nn.init.kaiming_normal_",       "Part 2"),
    ("AdamW optimiser",              "optim.AdamW",                   "Part 2"),
    ("Cosine annealing LR",          "CosineAnnealingLR",             "Part 2"),
    ("Dropout(0.3)",                 "nn.Dropout",                    "Part 2"),
    ("Early stopping (patience=5)",  "val_loss tracking",             "Part 2"),
    ("Confusion matrix",             "sklearn.metrics",               "Part 3"),
    ("Per-class F1 / recall",        "classification_report",         "Part 3"),
    ("Error analysis",               "misclassified visualisation",   "Part 3"),
    ("ONNX export",                  "torch.onnx.export",             "Part 4"),
    ("Inference profiling",          "throughput / latency / FLOPs",  "Part 4"),
]
tech_df = pd.DataFrame(techniques, columns=["Technique", "Implementation", "Where"])
print("\nFull technique inventory:")
print(tech_df.to_string(index=False))


---
## Part 5 — Summary & Key Takeaways

### Key Takeaways

- **Architecture search** before committing to full training avoids wasting compute
  on suboptimal designs.  A 3-epoch proxy trial on 10 K samples is cheap and
  highly predictive of final accuracy.

- **The complete toolkit matters**: combining He init + AdamW + cosine LR +
  Dropout + early stopping consistently outperforms any single technique alone.
  Each component targets a different failure mode (symmetry, weight explosion,
  LR decay, co-adaptation, overfitting respectively).

- **Confusion matrices and per-class F1** reveal which classes are genuinely hard
  for the model (Shirt vs T-shirt/Pullover) versus which are just slightly
  under-represented — a crucial distinction for production debugging.

- **ONNX export** decouples the trained model from the Python/PyTorch ecosystem.
  A model exported once can be served at low latency across runtimes, devices,
  and programming languages.

- **Inference profiling** (FLOPs, throughput, latency) bridges research and
  deployment — a model that is 0.2% more accurate but 10× slower is often the
  *wrong* trade-off for production.

### What's Next

→ **Module 06 (CNNs)** replaces the dense layers with convolutional filters,
adding spatial invariance and reducing the parameter count drastically — the same
FashionMNIST task can reach > 93 % with a simple CNN.

### Going Further

- **Batch normalisation** (05-05) can be enabled in `FlexMLP` via `use_batchnorm=True`
  and often improves accuracy by 1–2 pp on deeper networks.
- **Learning rate warmup** (linear ramp + cosine decay) is standard for Transformers
  and is covered in Module 16.
- **Mixed-precision training** (`torch.cuda.amp`) halves memory and doubles
  throughput on modern GPUs (Module 16).
- LeCun et al., *Efficient BackProp*, 1998 — the origin of normalisation and init best practices.
